# ROGII Wellbore TVT — **Stacked-UNet** Inference · **K=3 refeed** · **S2 SPATIAL CHANNEL + GATE (the 5.0270 line)** · PhaseJitter TTA

The §37.8 no-decay gate line, deployed. The spatial-neighbor prior enters **in-model** exactly as in
training/eval: rendered as the 6th input channel (Gaussian belief bump), consumed by stage 2 under the
bump-entry schedule (ENTRY=2, decay=1.0 → rounds [0, 1, 1, 1]), decoded through the learned **gate**
(blend-as-a-layer) at every refeed depth. Donors = ALL train wells (the deploy configuration; the
fold-honest CV price used ~620/fold). **The post-hoc DO_SPATIAL_BLEND wiring is REMOVED** — the model
LEARNED the signal; there is no artifact gate, no u-calibration, no per-seed streams. PhaseJitter TTA
is unchanged (full-bin 8-offset sweep; the spatial channel re-renders per phase crop like every other
channel).


In [ ]:
!pip install -q /kaggle/input/datasets/tuckerarrants/smp-offline/smp_offline_wheels/smp_offline_wheels/segmentation_models_pytorch-0.5.0-py3-none-any.whl

In [ ]:
!pip install -q /kaggle/input/datasets/tuckerarrants/smp-offline/smp_offline_wheels/smp_offline_wheels/timm-1.0.27-py3-none-any.whl

In [ ]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp

# Match the FT26 manifest exactly (these are the locked constants)
H = 256
W = 512
BIN_SIZE = 1.0
K_MAX = 12
ENCODER_NAME = 'tu-tf_efficientnetv2_xl.in21k_ft_in1k'
IN_CH_BASE = 5
IN_CH = 6            # base 5 + the S2 spatial bump channel (channel idx 5)
# --- Stacked-model fork knobs ---
STACK_N      = 2     # 2 = StackedUNet (stage1->posterior->stage2); 1 = single UNetGeneric (inert, == original kernel)
STACK_DETACH = True  # matches training; numerically INERT at inference (no_grad makes .detach() a no-op) — used only to build the arch
CKPT_DEVICE  = None  # None -> DEVICE (all ckpts resident on GPU, original behaviour). Set 'cpu' to stream each model GPU<-CPU per forward if a 16GB T4 OOMs on 25x stacked-B7 (~13.5GB fp32).

# Locate competition data — try both common Kaggle path conventions
CANDIDATE_COMP_ROOTS = [
    '/kaggle/input/rogii-wellbore-geology-prediction',
    '/kaggle/input/competitions/rogii-wellbore-geology-prediction',
]
COMP_ROOT = None
for cand in CANDIDATE_COMP_ROOTS:
    if os.path.exists(cand):
        COMP_ROOT = cand
        break
if COMP_ROOT is None:
    raise FileNotFoundError(f'Could not find competition data. Tried: {CANDIDATE_COMP_ROOTS}')
print(f'Competition data at: {COMP_ROOT}')
TEST_DIR  = os.path.join(COMP_ROOT, 'test')
SAMPLE_SUB = os.path.join(COMP_ROOT, 'sample_submission.csv')

# === Locate the Stacked-UNet artifact folder (ft39l) ===
# When you upload the FT26 artifact folder to Kaggle as a dataset, Kaggle assigns
# a slug like /kaggle/input/<dataset-name>/. The dataset must contain:
#   - ft39l_kaggle_inference_manifest.json
#   - ckpt_f{0..4}_s{seeds}.pt  (25 files: 5 folds x 5 seeds, EMA-shadow StackedUNet state_dicts)
#
# Auto-locate by scanning /kaggle/input for any directory containing the manifest.
CKPT_DIR = '/kaggle/input/datasets/tuckerarrants/rogii-unet/inference'
for dataset_name in sorted(os.listdir('/kaggle/input')):
    candidate = f'/kaggle/input/{dataset_name}'
    manifest_candidate = os.path.join(candidate, 'ft39l_kaggle_inference_manifest.json')
    if os.path.exists(manifest_candidate):
        CKPT_DIR = candidate
        break

if CKPT_DIR is None:
    available = os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else ['(directory missing)']
    raise FileNotFoundError(
        f'Could not find the Stacked-UNet artifact folder under /kaggle/input.\n'
        f'Looking for a directory containing ft39e_kaggle_inference_manifest.json.\n'
        f'Available /kaggle/input/ entries: {available}\n'
        f'Make sure you have uploaded the ft39e dz-channel artifact folder as a Kaggle dataset\n'
        f'and attached it to this notebook.'
    )

print(f'Found Stacked-UNet artifacts at: {CKPT_DIR}')
print(f'Contents (first 5): {sorted(os.listdir(CKPT_DIR))[:5]}')
print(f'  ... ({len(os.listdir(CKPT_DIR))} files total)')

# === Device discovery + ensemble sharding ===
# Kaggle's "2x T4" exposes two CUDA devices (cuda:0 + cuda:1) = 32 GB total, but the original kernel
# only ever used cuda:0 -> all checkpoints competed for a single 16 GB card (~0.54 GB/ckpt fp32:
# 25x barely fits, 35x OOMs). Spread the ensemble across every visible GPU so 35x stacked-B7
# (~18.8 GB fp32) lives across both cards. Numerically IDENTICAL to single-GPU fp32: a checkpoint on
# cuda:1 produces the same logits it would on cuda:0 (no precision change, CV stays bit-faithful).
N_GPU  = torch.cuda.device_count() if torch.cuda.is_available() else 0
GPUS   = [torch.device(f'cuda:{i}') for i in range(N_GPU)] if N_GPU else [torch.device('cpu')]
DEVICE = GPUS[0]                                  # inputs / intermediates default here
CKPT_DEVICE = DEVICE if CKPT_DEVICE is None else torch.device(CKPT_DEVICE)
# Sharding and cpu-streaming are mutually-exclusive residency strategies: only shard when we have
# >=2 GPUs AND the user hasn't requested streaming (CKPT_DEVICE left == compute device).
SHARD_MODELS = (N_GPU >= 2) and (str(CKPT_DEVICE) == str(DEVICE))
print(f'Visible GPUs: {N_GPU} -> {GPUS}')
print(f'Compute device: {DEVICE} | shard ensemble across GPUs: {SHARD_MODELS}')
print(f'Checkpoint residency mode: {CKPT_DEVICE}  (STACK_N={STACK_N})')

# === FORK KNOBS: K-refeed inference (spatial channel is ALWAYS ON for this artifact) ==========
EVAL_K = 3            # refeed depth. 3 = deployment (training CHECKPOINT_K).
assert STACK_N >= 2 or EVAL_K == 1, 'refeed iteration needs the stacked model'
K_COMPUTE = EVAL_K    # no blend streams anymore -> nothing beyond the deploy depth is needed

# === S2 SPATIAL CHANNEL + SCHEDULE + GATE — constants VERBATIM from the training config =======
# (2SUNET_BF16_S2HM_SCHED_GATE, the 5.0270 line; verify against ft39l_summary.json `experiment`.)
SPATIAL_PRIOR_ENABLE = True
SPRIOR_DELIVERY = 'stage2_heatmap'
HM_SIGMA_A, HM_SIGMA_B = 3.0, 4.0
HM_SIGMA_MIN, HM_SIGMA_MAX = 2.0, 40.0
HM_TRUST_ENABLE = True
HM_TRUST_DD0 = 6.9
HM_TRUST_DD_W = 0.35
HM_AMP_FLOOR = 0.05
HM_REFEED_DECAY = 1.0     # OFF — the §37.8 win; schedule = [0, 1, 1, 1]
HM_BUMP_ENTRY = 2         # inversion: round 1 dark, prior arrives round 2
HM_R1_W = 1.0
GATE_ENABLE = True
GATE_MASS_EPS = 0.05
AUX_TRUST_ENABLE = True
HANDOFF_TRAIN_T = 1.0     # identity on this line — kept explicit so eval == training eval by reading
REFEED_TRAIN_T = 1.0
assert HM_BUMP_ENTRY >= 2 and EVAL_K >= HM_BUMP_ENTRY, 'ENTRY>=2 requires deploy k>=ENTRY (bump-blind otherwise)'

# donors for the spatial channel: the competition train/ folder (ALWAYS required now)
TRAIN_DIR_DONORS = os.path.join(COMP_ROOT, 'train')
assert os.path.isdir(TRAIN_DIR_DONORS), f'spatial channel needs train/ as the donor source; not found at {TRAIN_DIR_DONORS}'
print(f'EVAL_K={EVAL_K} | spatial CHANNEL: ON (in-model; schedule ENTRY={HM_BUMP_ENTRY}, decay={HM_REFEED_DECAY}) | gate: ON')


In [ ]:
# Load the manifest — this is the source of truth for which checkpoints to use
manifest_path = os.path.join(CKPT_DIR, 'ft39l_kaggle_inference_manifest.json')
with open(manifest_path, 'r') as fh:
    manifest = json.load(fh)

print(f'Experiment: {manifest["experiment_id"]}')
print(f'N checkpoints: {manifest["n_checkpoints"]}')
print(f'Full-CV ens TVT-RMSE (training estimate): {manifest["full_cv_metric"]["ens_rmse"]:.3f}')
print(f'Recipe encoder: {manifest["recipe"]["encoder_name"]}, in_ch: {manifest["recipe"]["in_ch"]}')

# Verify constants in manifest match the constants we set above
assert manifest['image_constants']['H'] == H
assert manifest['image_constants']['W'] == W
assert manifest['image_constants']['BIN_SIZE'] == BIN_SIZE
assert manifest['image_constants']['K_MAX'] == K_MAX
assert manifest['recipe']['encoder_name'] == ENCODER_NAME
assert manifest['recipe']['in_ch'] == IN_CH
# Stacked-model awareness (fork): the artifact must declare the same stack depth as the kernel.
_stack_n_manifest = int(manifest['recipe'].get('stack_n', 1))
assert _stack_n_manifest == STACK_N, (
    f'Manifest stack_n={_stack_n_manifest} but kernel STACK_N={STACK_N}. '
    f'Set STACK_N in the config cell to match the uploaded artifact.')
if STACK_N >= 2:
    print(f"Stacked artifact: stack_n={_stack_n_manifest}, "
          f"handoff={manifest['recipe'].get('stack_handoff')}, stage2_in_ch={IN_CH + 1}")
print(f'N checkpoints in manifest: {manifest["n_checkpoints"]}')
print('Manifest constants match locally hardcoded constants — OK')


In [ ]:
# === DECODER_INTERP: pixelshuffle-checkpoint swap ============================================
# The downloaded models graft a learned sub-pixel upsampler onto each SMP decoder block. Inference
# MUST rebuild the SAME surgery, or load_state_dict sees unexpected "subpixel_upsamplers.*" keys and
# cell 7 fails loud. Set this to match the checkpoint set being loaded:
#   "nearest"/"bilinear"  = native SMP interp, no subpixel module   [original baseline]
#   "pixelshuffle"        = bilinear base + zero-init 1x1 sub-pixel residual   [the 5.8-CV set]
#   "pixelshuffle_icnr"   = pure ICNR 3x3 sub-pixel upsampler (no bilinear base)
DECODER_INTERP = "nearest"     # was "pixelshuffle"; the dilated-128 member uses the BASELINE decoder

# === POSTERIOR MD-DILATION: match the dilated checkpoint set ===================================
# The dilated models insert a residual MD-only dilated 1-D stack on the stage-1 posterior plane
# (between the two stages). Inference MUST rebuild it, or cell 7 sees unexpected "post_md_dilation.*"
# keys (POSTERIOR_DILATION too low) / leaves them zero-init (too high). Set to match the ckpt set:
#   POSTERIOR_DILATION=False                            -> baseline / pixelshuffle sets (no dilation)
#   POSTERIOR_DILATION=True, EMBED=128, PARALLEL=False   -> the serial dilated-128 member  [this set]
POSTERIOR_DILATION = False
POST_DIL_EMBED     = 128
POST_DIL_RATES     = (1, 2, 4, 8, 16)
POST_DIL_PARALLEL  = False

# --- sub-pixel decoder helpers (VERBATIM from training cell 16; freeze omitted - inference-irrelevant) ---
def _smp_interp_mode(m):
    # "pixelshuffle" is not a native F.interpolate mode -> build SMP with bilinear as the residual base.
    return "bilinear" if m in ("pixelshuffle", "pixelshuffle_icnr") else m

def _make_subpixel(c, r=2):
    conv = nn.Conv2d(c, c * r * r, kernel_size=1)          # 1x1: minimal added capacity
    nn.init.zeros_(conv.weight); nn.init.zeros_(conv.bias)  # zero-init -> residual == 0 at start (== bilinear base)
    return nn.Sequential(conv, nn.PixelShuffle(r))

def _icnr_(weight, r=2, init=nn.init.kaiming_normal_):
    # ICNR (Aitken 2017): init sub-pixel conv so PixelShuffle(r) == nearest upsample at start (checkerboard-free).
    oc, ic, kh, kw = weight.shape
    sub = torch.zeros(oc // (r * r), ic, kh, kw)
    init(sub)
    weight.data.copy_(sub.repeat_interleave(r * r, dim=0))   # replicate each sub-kernel r*r times

def _make_subpixel_icnr(c, r=2):
    conv = nn.Conv2d(c, c * r * r, kernel_size=3, padding=1)   # 3x3: learns the FULL upsampling (no bilinear base)
    _icnr_(conv.weight, r); nn.init.zeros_(conv.bias)          # ICNR -> == nearest upsample at start
    return nn.Sequential(conv, nn.PixelShuffle(r))

def _add_pixelshuffle_residual(unet, in_ch, _probe_hw=(256, 512)):
    """Add a zero-init sub-pixel residual to each decoder block upsample:
    feature_map <- bilinear_up(fm) + PixelShuffle(conv1x1(fm)). Zero-init => starts EXACTLY at the
    bilinear baseline and learns sharpening on top. SMP has no hook, so: discover each block input-ch
    via a dummy forward, build one subpixel module per block (registered -> params train + saved in
    pretrain state), and wrap block.forward (reproducing SMP UnetDecoderBlock.forward) to add the residual."""
    blocks = unet.decoder.blocks
    in_ch_per = [None] * len(blocks); handles = []
    for i, b in enumerate(blocks):
        def _hook(mod, args, _i=i):
            fm = args[0] if args else None
            in_ch_per[_i] = fm.shape[1] if torch.is_tensor(fm) else None
        handles.append(b.register_forward_pre_hook(_hook))
    was = unet.training; unet.eval()
    with torch.no_grad():
        unet(torch.zeros(1, in_ch, *_probe_hw))
    unet.train(was)
    for h in handles: h.remove()
    subs = nn.ModuleList([_make_subpixel(c) if c else nn.Identity() for c in in_ch_per])
    unet.add_module("subpixel_upsamplers", subs)
    n = 0
    for b, sub in zip(blocks, subs):
        if isinstance(sub, nn.Identity): continue
        n += 1
        def _patched(feature_map, target_height, target_width, skip_connection=None, _b=b, _sub=sub, _mode=b.interpolation_mode):
            up = F.interpolate(feature_map, size=(target_height, target_width), mode=_mode)
            res = _sub(feature_map)
            if res.shape[-2:] != up.shape[-2:]:
                res = F.interpolate(res, size=(target_height, target_width), mode="bilinear", align_corners=False)
            feature_map = up + res
            if skip_connection is not None:
                feature_map = torch.cat([feature_map, skip_connection], dim=1)
                feature_map = _b.attention1(feature_map)
            feature_map = _b.conv1(feature_map)
            feature_map = _b.conv2(feature_map)
            feature_map = _b.attention2(feature_map)
            return feature_map
        b.forward = _patched
    assert n > 0, "DECODER_INTERP='pixelshuffle' but no decoder blocks found to augment"
    print(f"  [PIXELSHUFFLE] zero-init 1x1 subpixel residual on {n} block(s); in-ch = {[c for c in in_ch_per if c]}")
    return unet

def _add_pixelshuffle_icnr(unet, in_ch, _probe_hw=(256, 512)):
    """PURE pixel-shuffle upsampler (NO bilinear base): feature_map <- PixelShuffle(conv3x3(fm)), ICNR-init
    so it == nearest upsample at start, then learns the entire upsampling. Decouples from the bilinear anchor.
    Same machinery as the residual variant; registers under the same 'subpixel_upsamplers' name."""
    blocks = unet.decoder.blocks
    in_ch_per = [None] * len(blocks); handles = []
    for i, b in enumerate(blocks):
        def _hook(mod, args, _i=i):
            fm = args[0] if args else None
            in_ch_per[_i] = fm.shape[1] if torch.is_tensor(fm) else None
        handles.append(b.register_forward_pre_hook(_hook))
    was = unet.training; unet.eval()
    with torch.no_grad():
        unet(torch.zeros(1, in_ch, *_probe_hw))
    unet.train(was)
    for h in handles: h.remove()
    subs = nn.ModuleList([_make_subpixel_icnr(c) if c else nn.Identity() for c in in_ch_per])
    unet.add_module("subpixel_upsamplers", subs)
    n = 0
    for b, sub in zip(blocks, subs):
        if isinstance(sub, nn.Identity): continue
        n += 1
        def _patched(feature_map, target_height, target_width, skip_connection=None, _b=b, _sub=sub):
            up = _sub(feature_map)                                  # pure pixel shuffle, NO bilinear base
            if up.shape[-2:] != (target_height, target_width):
                up = F.interpolate(up, size=(target_height, target_width), mode="bilinear", align_corners=False)
            feature_map = up
            if skip_connection is not None:
                feature_map = torch.cat([feature_map, skip_connection], dim=1)
                feature_map = _b.attention1(feature_map)
            feature_map = _b.conv1(feature_map)
            feature_map = _b.conv2(feature_map)
            feature_map = _b.attention2(feature_map)
            return feature_map
        b.forward = _patched
    assert n > 0, "DECODER_INTERP='pixelshuffle_icnr' but no decoder blocks found to augment"
    print(f"  [PIXELSHUFFLE-ICNR] pure ICNR 3x3 upsampler on {n} block(s); in-ch = {[c for c in in_ch_per if c]}")
    return unet


class UNetGeneric(nn.Module):
    '''Thin wrapper around smp.Unet. State-dict keys are prefixed with smp. (e.g. smp.encoder.conv1.weight).
    This is the single-UNet (STACK_N==1) inert path — bit-identical to the pre-fork kernel.'''
    def __init__(self, encoder_name='resnet34', in_ch=5):
        super().__init__()
        # encoder_weights=None: we load trained weights; Kaggle runs offline so no ImageNet download.
        self.smp = smp.Unet(encoder_name=encoder_name, encoder_weights=None,
                            in_channels=in_ch, classes=1,
                            decoder_interpolation=_smp_interp_mode(DECODER_INTERP))
        if DECODER_INTERP == "pixelshuffle":
            _add_pixelshuffle_residual(self.smp, in_ch)
        elif DECODER_INTERP == "pixelshuffle_icnr":
            _add_pixelshuffle_icnr(self.smp, in_ch)
    def forward(self, x):
        return self.smp(x).squeeze(1)


class PosteriorMDDilation(nn.Module):
    """Inference twin of the training PosteriorMDDilation. Residual MD-only dilated 1-D stack on the
    stage-1 posterior plane (rows-as-channels, MD as conv axis). Serial cascade by default; parallel
    multi-rate concat when POST_DIL_PARALLEL. Output projection zero-init. State-dict keys
    (post_md_dilation.proj_in / trunk|branches+mix / proj_out) MUST match the dilated checkpoints."""
    def __init__(self, n_rows=H, embed=POST_DIL_EMBED, dilations=POST_DIL_RATES, k=3, parallel=None):
        super().__init__()
        self.parallel = POST_DIL_PARALLEL if parallel is None else parallel
        self.proj_in = nn.Conv1d(n_rows, embed, kernel_size=1)
        if self.parallel:
            nbr = len(dilations); base = embed // nbr
            widths = [base] * (nbr - 1) + [embed - base * (nbr - 1)]
            self.branches = nn.ModuleList([
                nn.Conv1d(embed, w, k, dilation=d, padding=((k - 1) // 2) * d) for d, w in zip(dilations, widths)])
            self.mix = nn.Conv1d(embed, embed, kernel_size=1); self.act = nn.GELU()
        else:
            trunk = []
            for d in dilations:
                trunk += [nn.Conv1d(embed, embed, k, dilation=d, padding=((k - 1) // 2) * d), nn.GELU()]
            self.trunk = nn.Sequential(*trunk)
        self.proj_out = nn.Conv1d(embed, n_rows, kernel_size=1)
        nn.init.zeros_(self.proj_out.weight); nn.init.zeros_(self.proj_out.bias)
    def forward(self, post1):
        z = self.proj_in(post1)
        if self.parallel:
            z = self.act(self.mix(torch.cat([br(z) for br in self.branches], dim=1)))
        else:
            z = self.trunk(z)
        delta = self.proj_out(z)
        return post1 + delta


def _hm_round_weight(round_idx):
    """S2HM bump-entry schedule (SCHED arm). Rounds before HM_BUMP_ENTRY are DARK (weight 0);
    from ENTRY the geometric chain runs: w(r) = HM_REFEED_DECAY**(r - HM_BUMP_ENTRY). HM_R1_W
    additionally scales round 1 only (middle-schedule branch; moot when ENTRY >= 2).
    IDENTITY: ENTRY=1 & R1_W=1.0 -> [1, d, d^2, ...] == the DECAY arm bit-identically.
    INVERSION: ENTRY=2 -> [0, 1, d, d^2, ...] (round 1 dark; prior = second opinion)."""
    w = 0.0 if round_idx < HM_BUMP_ENTRY else float(HM_REFEED_DECAY) ** (round_idx - HM_BUMP_ENTRY)
    if round_idx == 1:
        w *= float(HM_R1_W)
    return w

_sched_chk = [_hm_round_weight(r) for r in (1, 2, 3, 4)]
assert _sched_chk == [0.0, 1.0, 1.0, 1.0], f'schedule mismatch for the 5.0270 line: {_sched_chk}'
print(f'S2HM schedule (rounds 1..4): {_sched_chk}  [ENTRY={HM_BUMP_ENTRY}, decay={HM_REFEED_DECAY}]')

def _gate_compute(gate_head, gate_alpha, bump_raw, logits2, aux_trust=None, mass_eps=None):
    """§GATE core (module-level so the smoke exercises the EXACT deployed math standalone).
    bump_raw: (B,H,W) RAW rendered bump channel — schedule-independent (the schedule scales the
    model-INPUT copy inside forward; x itself always carries the render), and it rides every aug,
    so the derived prior row is warp/flip-consistent by construction. logits2: (B,H,W).
    aux_trust: (B,W) in [0,1] or None (-> 0.5 constant). ALL trunk quantities are detached HERE —
    the gate is a PURE COMBINER; its gradient reaches only gate_head/gate_alpha.
    Returns (g, prior_row, valid, ent), each (B, W). Invalid (no-prior) columns: g=0 AND
    prior_row=E[row] (belt+braces: pred == E[row] regardless of either)."""
    eps = GATE_MASS_EPS if mass_eps is None else mass_eps
    Hn = logits2.size(1)
    rows = torch.arange(Hn, device=logits2.device, dtype=torch.float32)
    lg = logits2.detach()
    p = F.softmax(lg, dim=1)
    er = (p * rows[None, :, None]).sum(dim=1)
    ent = -(p.clamp_min(1e-9).log() * p).sum(dim=1) / float(np.log(Hn))
    pmax = p.max(dim=1).values
    bump = bump_raw.detach().clamp_min(0.0)
    mass = bump.sum(dim=1)
    valid = mass > eps
    prow = (bump * rows[None, :, None]).sum(dim=1) / mass.clamp_min(1e-9)
    peak = bump.max(dim=1).values
    agree = (er - prow).abs() / float(Hn)
    trust = aux_trust.detach() if aux_trust is not None else torch.full_like(er, 0.5)
    feats = torch.stack([ent, pmax, agree, trust, peak, (mass / 10.0).clamp(0.0, 2.0)], dim=-1)
    g = gate_alpha * torch.sigmoid(gate_head(feats).squeeze(-1))
    g = torch.where(valid, g, torch.zeros_like(g))
    prow = torch.where(valid, prow, er)
    return g, prow, valid, ent

class StackedUNet(nn.Module):
    '''Two-stage StackedUNet — inference twin of the TRAINING class (2SUNET_BF16_S2HM_SCHED_GATE
    cell 22), spatial-channel edition. Stage 1 sees the 5 BASE channels only; the bump (input
    channel IN_CH_BASE) is injected into STAGE 2 next to the refed posterior, scaled per round by
    _hm_round_weight (exact-zero branch at w=0). State-dict keys stage1.*/stage2.*/aux_head.*/
    gate_head.*/gate_alpha MUST match the training class for the EMA-shadow checkpoints to load.
    detach flags are value-inert under no_grad.'''
    def __init__(self, encoder_name='efficientnet-b7', in_ch=6, detach_handoff=True):
        super().__init__()
        self.detach_handoff = detach_handoff
        self._hm = (SPATIAL_PRIOR_ENABLE and SPRIOR_DELIVERY == 'stage2_heatmap')
        s1_in = IN_CH_BASE if self._hm else in_ch
        self.stage1 = smp.Unet(encoder_name=encoder_name, encoder_weights=None,
                               in_channels=s1_in, classes=1,
                               decoder_channels=(512, 256, 128, 64, 32),
                               decoder_interpolation=_smp_interp_mode(DECODER_INTERP))
        self.stage2 = smp.Unet(encoder_name=encoder_name, encoder_weights=None,
                               in_channels=(IN_CH_BASE + 2) if self._hm else (in_ch + 1), classes=1,
                               decoder_channels=(512, 256, 128, 64, 32),
                               decoder_interpolation=_smp_interp_mode(DECODER_INTERP))
        if DECODER_INTERP == "pixelshuffle":
            _add_pixelshuffle_residual(self.stage1, s1_in)
            _add_pixelshuffle_residual(self.stage2, (IN_CH_BASE + 2) if self._hm else (in_ch + 1))
        elif DECODER_INTERP == "pixelshuffle_icnr":
            _add_pixelshuffle_icnr(self.stage1, s1_in)
            _add_pixelshuffle_icnr(self.stage2, (IN_CH_BASE + 2) if self._hm else (in_ch + 1))
        if POSTERIOR_DILATION:
            self.post_md_dilation = PosteriorMDDilation()
        if AUX_TRUST_ENABLE and self._hm:
            self.aux_head = nn.Sequential(nn.Conv1d(H, 64, 1), nn.GELU(), nn.Conv1d(64, 1, 1))
        if GATE_ENABLE and self._hm:
            self.gate_head = nn.Sequential(nn.Linear(6, 32), nn.GELU(), nn.Linear(32, 1))
            self.gate_alpha = nn.Parameter(torch.zeros(1))

    def gate_g(self, x, logits2):
        '''Per-column blend gate — VERBATIM contract from training (aux trust detached feature).'''
        at = None
        if getattr(self, 'aux_head', None) is not None:
            with torch.no_grad():
                at = torch.sigmoid(self.aux_head(logits2.detach()).squeeze(1))
        return _gate_compute(self.gate_head, self.gate_alpha, x[:, IN_CH_BASE], logits2, aux_trust=at)

    def _stage2_from_post(self, x, post, round_idx=1):
        '''One stage-2 pass; bump slice scaled by _hm_round_weight(round_idx) — the SCHED schedule,
        exact-zero branch (junk-proof at w=0). VERBATIM logic from training.'''
        if self._hm:
            _w = _hm_round_weight(round_idx)
            _b = x[:, IN_CH_BASE:IN_CH_BASE + 1]
            bump = _b * _w if _w != 0.0 else torch.zeros_like(_b)
            x2 = torch.cat([x[:, :IN_CH_BASE], post.unsqueeze(1), bump], dim=1)
        else:
            x2 = torch.cat([x, post.unsqueeze(1)], dim=1)
        return self.stage2(x2).squeeze(1)

    def forward(self, x, return_aux=False):
        x1 = x[:, :IN_CH_BASE] if self._hm else x
        logits1 = self.stage1(x1).squeeze(1)
        handoff = logits1.detach() if self.detach_handoff else logits1
        post1 = F.softmax(handoff / HANDOFF_TRAIN_T, dim=1)
        if POSTERIOR_DILATION:
            post1 = self.post_md_dilation(post1)
        logits2 = self._stage2_from_post(x, post1, round_idx=1)
        if return_aux:
            return logits2, logits1
        return logits2


def build_model(encoder_name=ENCODER_NAME, in_ch=IN_CH):
    '''Single source of truth for inference model construction (reads STACK_N at call time).'''
    if STACK_N >= 2:
        return StackedUNet(encoder_name=encoder_name, in_ch=in_ch, detach_handoff=STACK_DETACH)
    return UNetGeneric(encoder_name=encoder_name, in_ch=in_ch)


def _iter_logits(model, x, k):
    '''k-round refeed iteration — mirrors the TRAINING _eval_iter_logits exactly: stage-1 ->
    handoff (/HANDOFF_TRAIN_T) -> stage-2 via model._stage2_from_post(round_idx=1), then re-feed
    softened by REFEED_TRAIN_T for rounds 2..k, each with its own round_idx so the bump schedule
    applies (dark at round 1 under ENTRY=2, full from round 2 at decay=1.0). Returns {j: logits2}.'''
    if STACK_N < 2:
        assert k == 1, 'single-UNet path supports k=1 only'
        return {1: model(x)}
    logits1 = model.stage1(x[:, :IN_CH_BASE] if getattr(model, '_hm', False) else x).squeeze(1)
    handoff = logits1.detach() if model.detach_handoff else logits1
    post = F.softmax(handoff / HANDOFF_TRAIN_T, dim=1)
    if POSTERIOR_DILATION:
        post = model.post_md_dilation(post)
    out = {}
    logits2 = model._stage2_from_post(x, post, round_idx=1)
    out[1] = logits2
    for j in range(2, k + 1):
        post = F.softmax(logits2 / REFEED_TRAIN_T, dim=1)
        logits2 = model._stage2_from_post(x, post, round_idx=j)
        out[j] = logits2
    return out


_row_idx_cache = {}
def _row_idx_for(dev):
    t = _row_idx_cache.get(dev)
    if t is None:
        t = torch.arange(H, device=dev, dtype=torch.float32)
        _row_idx_cache[dev] = t
    return t

def _gated_expected_row(model, x, logits2):
    '''Deployment decode — VERBATIM from the training evaluate_model: softmax-expected row, then
    the learned gate blends toward the prior row on valid (bump-mass) columns. (B, W) float.'''
    ri = _row_idx_for(logits2.device)
    er = (F.softmax(logits2, dim=1) * ri[None, :, None]).sum(dim=1)
    if GATE_ENABLE and SPATIAL_PRIOR_ENABLE and getattr(model, '_hm', False):
        g, prow, valid, _ = model.gate_g(x, logits2)
        er = torch.where(valid, (1.0 - g) * er + g * prow, er)
    return er


In [ ]:
# === dZ-per-column channel helpers (FT39e) — VERBATIM copy from training cell 12 ===
def _dz_per_col(hw_z, hw_md, bin_edges, hw_mask):
    """Per-column average of dZ/dMD. Returns float32 (W,) with NaN where no data."""
    n = len(hw_z)
    if n < 2:
        return np.full(W, 0.0, dtype=np.float32)
    # uniform 1.0 ft MD step per the EDA report, but compute defensively
    dz_per_row = np.gradient(np.asarray(hw_z, dtype=np.float64),
                              np.asarray(hw_md, dtype=np.float64))
    dz = np.full(W, np.nan, dtype=np.float32)
    for c in range(W):
        lo, hi = bin_edges[c], bin_edges[c+1]
        if hi <= lo:
            continue
        dz[c] = float(np.mean(dz_per_row[lo:hi]))
    return dz

def _dz_channel(dz_per_col, hw_mask):
    """Build the (H, W) dZ channel — per-well z-score + clip ±3 + broadcast across H.
    Neutralize columns without HW data so they don't bias the well z-score / clip."""
    bad = (~hw_mask) | (~np.isfinite(dz_per_col))
    if (~bad).any():
        fill = float(np.nanmean(dz_per_col[~bad]))
    else:
        fill = 0.0
    dz = np.where(bad, fill, dz_per_col).astype(np.float32)
    m, s = dz.mean(), dz.std()
    dz_norm = (dz - m) / s if s > 1e-6 else np.zeros_like(dz)
    dz_norm = np.clip(dz_norm, -3.0, 3.0)
    return np.broadcast_to(dz_norm[None, :], (H, W)).copy().astype(np.float32)


def build_image_from_arrays(hw_md, hw_gr, hw_z, hw_tvt, hw_tvti, tw_tvt, tw_gr,
                            sp_prior_tvt=None, sp_logdd=None):
    '''Identical to FT39e cell 12 build_image_from_arrays (5-channel incl. dZ).

    For test wells, hw_tvt is all NaN (target not provided).
    The function uses hw_tvt only to compute truth_row, which is not needed at inference.

    SPATIAL CHANNEL (the 5.0270 line): sp_prior_tvt / sp_logdd are ROW-level (n_hw,) arrays — the
    engine's prior TVT per post-PS row (NaN elsewhere) and log1p(donor distance) per row. They are
    binned to columns HERE (mean of finite values per column — the training _sp_bin_to_cols math),
    converted to grid rows against THIS render's tvt_lo, and painted as channel 5: an amp(dd)-scaled
    Gaussian bump with sigma(dd), post-PS columns only (training make_jittered_image at k=0). Being
    row-level inputs, they truncate with the PhaseJitter toe-crop like every other array, so the
    bump re-renders consistently per phase offset. None -> zero channel (the training convention
    for wells without a prior: zero bump == frozen baseline via the zero-init stem slice).
    '''
    n_hw = len(hw_md)
    known_tvt = pd.Series(hw_tvti).dropna().values
    if len(known_tvt) == 0:
        raise ValueError('No pre-PS rows')
    depth_center = float(known_tvt[-1])
    tvt_lo = depth_center - (H/2) * BIN_SIZE

    def tvt_to_row(t, lo=tvt_lo):
        if np.isnan(t): return -1
        r = int((t - lo) / BIN_SIZE)
        return r if 0 <= r < H else -1

    bin_edges = np.linspace(0, n_hw, W + 1).astype(int)
    hw_gr_per_col_raw = np.full(W, np.nan, dtype=np.float32)
    hw_mask = np.zeros(W, dtype=bool)
    truth_row = np.full(W, -1, dtype=np.int32)
    hint_row = np.full(W, -1, dtype=np.int32)
    post_ps_mask = np.zeros(W, dtype=bool)

    for c in range(W):
        lo, hi = bin_edges[c], bin_edges[c+1]
        if hi <= lo: continue
        gr_slc = hw_gr[lo:hi]
        if np.any(~np.isnan(gr_slc)):
            hw_gr_per_col_raw[c] = np.nanmean(gr_slc)
            hw_mask[c] = True
        tvt_slc = hw_tvt[lo:hi]
        if np.any(~np.isnan(tvt_slc)):
            truth_row[c] = tvt_to_row(np.nanmedian(tvt_slc))
        tvti_slc = hw_tvti[lo:hi]
        is_known = ~np.isnan(tvti_slc)
        if is_known.any():
            hint_row[c] = tvt_to_row(np.nanmedian(tvti_slc[is_known]))
        else:
            post_ps_mask[c] = True

    if np.all(np.isnan(hw_gr_per_col_raw)):
        hw_gr_per_col = np.zeros(W, dtype=np.float32)
    else:
        col_mean = np.nanmean(hw_gr_per_col_raw)
        hw_gr_per_col = hw_gr_per_col_raw.copy()
        hw_gr_per_col[np.isnan(hw_gr_per_col)] = col_mean

    ext_row_tvts = (tvt_lo - K_MAX * BIN_SIZE) + (np.arange(H + 2*K_MAX) + 0.5) * BIN_SIZE
    sort_idx = np.argsort(tw_tvt)
    tw_tvt_s = tw_tvt[sort_idx]; tw_gr_s = tw_gr[sort_idx]
    ok = ~np.isnan(tw_gr_s)
    if ok.sum() < 2:
        tw_gr_per_row_ext = np.zeros(H + 2*K_MAX, dtype=np.float32)
    else:
        tw_gr_per_row_ext = np.interp(ext_row_tvts, tw_tvt_s[ok], tw_gr_s[ok],
                                       left=tw_gr_s[ok][0], right=tw_gr_s[ok][-1]).astype(np.float32)
    tw_gr_per_row = tw_gr_per_row_ext[K_MAX : K_MAX + H]

    ch_tw_raw = np.broadcast_to(tw_gr_per_row[:, None], (H, W)).copy()
    ch_hw_raw = np.broadcast_to(hw_gr_per_col[None, :], (H, W)).copy()
    ch_hint = np.zeros((H, W), dtype=np.float32)
    for c in range(W):
        if hint_row[c] >= 0 and not post_ps_mask[c]:
            ch_hint[hint_row[c], c] = 1.0

    cost_vol = (ch_hw_raw - ch_tw_raw) ** 2
    if hw_mask.any():
        mean_cost = float(cost_vol[:, hw_mask].mean())
    else:
        mean_cost = 0.0
    for c in range(W):
        if not hw_mask[c]:
            cost_vol[:, c] = mean_cost

    ch_tw = ch_tw_raw.copy()
    m, s = ch_tw.mean(), ch_tw.std()
    if s > 1e-6: ch_tw = (ch_tw - m) / s

    ch_hw = ch_hw_raw.copy()
    m, s = ch_hw.mean(), ch_hw.std()
    if s > 1e-6: ch_hw = (ch_hw - m) / s

    cm, cs = cost_vol.mean(), cost_vol.std()
    if cs > 1e-6:
        cost_vol_norm = (cost_vol - cm) / cs
    else:
        cost_vol_norm = cost_vol - cm
    cost_vol_norm = np.clip(cost_vol_norm, -3.0, 3.0)

    # --- dZ-per-column channel (FT39e) — VERBATIM stack order from training cell 12 ---
    dz_per_col = _dz_per_col(hw_z, hw_md, bin_edges, hw_mask)
    ch_dz = _dz_channel(dz_per_col, hw_mask)
    # --- channel 5: the S2 spatial belief bump (training make_jittered_image, k=0) ------------
    ch_bump = np.zeros((H, W), dtype=np.float32)
    if sp_prior_tvt is not None and sp_logdd is not None:
        # binning MATCHES training _sp_bin_to_cols exactly: float32 accumulators, 0.0 default
        # on empty columns (an empty post-PS column then lands far outside the paint window and
        # is dropped by okc — training's own behaviour; do NOT 'improve' this to NaN/fill).
        _pr_t = np.zeros(W, dtype=np.float32)             # binned prior TVT per column
        _ld = np.zeros(W, dtype=np.float32)               # binned log1p(dd) per column
        for c in range(W):
            lo, hi = bin_edges[c], bin_edges[c + 1]
            if hi <= lo: continue
            v = sp_prior_tvt[lo:hi]; v = v[np.isfinite(v)]
            if v.size: _pr_t[c] = float(v.mean())
            u = sp_logdd[lo:hi]; u = u[np.isfinite(u)]
            if u.size: _ld[c] = float(u.mean())
        pr = (_pr_t - tvt_lo) / BIN_SIZE                  # canonical grid row per column (k=0)
        pr = np.where(post_ps_mask, pr, np.nan)           # bump on post-PS columns only
        sg = np.clip(HM_SIGMA_A + HM_SIGMA_B * (_ld - HM_TRUST_DD0), HM_SIGMA_MIN, HM_SIGMA_MAX)
        sg = np.maximum(sg, 1e-3)
        if HM_TRUST_ENABLE:
            _t = 1.0 / (1.0 + np.exp((_ld - HM_TRUST_DD0) / HM_TRUST_DD_W))
            amp = np.clip(_t, HM_AMP_FLOOR, 1.0)
        else:
            amp = np.ones(W)
        okc = np.isfinite(pr) & post_ps_mask & (pr > -3 * sg) & (pr < H - 1 + 3 * sg)
        if okc.any():
            rowsg = np.arange(H, dtype=np.float32)[:, None]
            ch_bump[:, okc] = (amp[okc][None, :]
                               * np.exp(-(rowsg - pr[okc][None, :]) ** 2 / (2 * (sg[okc] ** 2)[None, :]))
                               ).astype(np.float32)
    image_in = np.stack([ch_tw, ch_hw, ch_hint, cost_vol_norm, ch_dz, ch_bump], axis=0).astype(np.float32)
    return dict(image_in=image_in, truth_row=truth_row, hint_row=hint_row,
                post_ps_mask=post_ps_mask, depth_center=depth_center,
                tvt_lo=tvt_lo, hw_mask=hw_mask, bin_edges=bin_edges, dz_per_col=dz_per_col)


In [ ]:
# === S2 SPATIAL CHANNEL — engine + donor field from ALL train wells (in-model delivery) ========
# The V0 drift-field engine, VERBATIM from the training notebook (cell §SP), constants from the
# training config (asserted there == the audited probe engine). Donors = every train well (the
# deploy configuration: 773 vs ~620/fold in CV -> the deployed prior is the STRONGER test-emulation
# one, 11.77 ft standalone vs 13.29 fold-clean). theta/kappa fit here from ALL train labels
# (legitimate at test time); kappa keeps per-donor self-exclusion. NO gate artifact, NO blend —
# the prior enters ONLY as the model's 6th input channel; the LEARNED in-model gate does the rest.
SP_K_SEG = 16; SP_RHO = 10.0; SP_LL_K = 50; SP_LL_H = 500.0; SP_RIDGE = 1.0
SP_FIELD_MIN_PROJ = 0.30; SP_GATE = 0.35
SP_KBINS = [0.0, 750.0, 1500.0, 2500.0, 4000.0, 1e18]; SP_NBIN = len(SP_KBINS) - 1

def _sp_seg_geom(X, Y, s, n, K=SP_K_SEG):
    """theta-independent geometry: segment ids, midpoints, azimuths."""
    edges = np.linspace(0, n, K + 1)
    segid = np.clip(np.searchsorted(edges[1:], np.arange(1, n + 1.0), side='left'), 0, K - 1)
    mid = np.empty((K, 2)); az = np.empty(K)
    for j in range(K):
        f0 = s + 1 + int(edges[j])
        f1 = min(s + 1 + max(int(edges[j + 1]) - 1, int(edges[j])), len(X) - 1)
        az[j] = np.arctan2(Y[f1] - Y[f0], X[f1] - X[f0])
        mid[j] = ((X[f0] + X[f1]) / 2.0, (Y[f0] + Y[f1]) / 2.0)
    return segid, mid, az

def _sp_hinge(n, K=SP_K_SEG):
    t = np.arange(1, n + 1.0); e = np.linspace(0, n, K + 1)
    return np.column_stack([np.clip(t - e[j], 0, e[j + 1] - e[j]) for j in range(K)])

def _sp_fit_c(drift, n, rho=0.0, K=SP_K_SEG):
    """per-segment drift SLOPE (ft per MD row) from a donor's true drift curve."""
    phi = _sp_hinge(n, K); A = phi.T @ phi
    if rho > 0:
        D = np.diff(np.eye(K), axis=0)
        A = A + rho * np.mean(np.diag(phi.T @ phi)) * D.T @ D
    return np.linalg.solve(A, phi.T @ drift)

def _sp_build_field(wids, key):
    """(x, y, TRUE dip = c / cos(az - theta), well_index). THE azimuth normalisation."""
    r = [(SP_WELL[w]['mid'][j, 0], SP_WELL[w]['mid'][j, 1],
          SP_WELL[w][key][j] / SP_WELL[w]['proj'][j], SP_WELL[w]['wi'])
         for w in wids for j in range(SP_K_SEG)
         if abs(SP_WELL[w]['proj'][j]) > SP_FIELD_MIN_PROJ]
    return np.asarray(r, dtype=np.float64)

def _sp_local_linear(field, mid, exclude_wi=()):
    keep = ~np.isin(field[:, 3], list(exclude_wi)) if len(exclude_wi) else np.ones(len(field), bool)
    fx, fy, fD = field[keep, 0], field[keep, 1], field[keep, 2]
    Dh = np.empty(len(mid)); dd = np.empty(len(mid))
    for j in range(len(mid)):
        d2 = (fx - mid[j, 0]) ** 2 + (fy - mid[j, 1]) ** 2
        k = min(SP_LL_K, len(d2) - 1)
        sel = np.argpartition(d2, k)[:k]
        w = np.exp(np.maximum(-d2[sel] / (2 * SP_LL_H ** 2), -700))
        dx = (fx[sel] - mid[j, 0]) / 1000.0; dy = (fy[sel] - mid[j, 1]) / 1000.0
        Xd = np.column_stack([np.ones(len(sel)), dx, dy])
        A = (Xd * w[:, None]).T @ Xd + SP_RIDGE * np.sum(w) * np.diag([0.0, 1.0, 1.0])
        Dh[j] = np.linalg.solve(A, (Xd * w[:, None]).T @ fD[sel])[0]
        dd[j] = np.sqrt(np.median(np.sort(d2[sel])[:15]))
    return Dh, dd

def _sp_design_T2(w, F_raw, F_sm, exclude_wi=()):
    """Design columns for the DRIFT ONLY. The flat -dZ baseline is exact physics, never fitted."""
    Dr, dd = _sp_local_linear(F_raw, w['mid'], exclude_wi)
    Ds, _  = _sp_local_linear(F_sm,  w['mid'], exclude_wi)
    ch_r = Dr * w['proj']; ch_s = Ds * w['proj']          # true dip -> THIS well's apparent dip
    gated = np.abs(w['proj'][w['segid']]) < SP_GATE       # running along strike: no dip info
    bstep = np.digitize(dd, SP_KBINS[1:-1])[w['segid']]
    pos = (w['segid'] + 0.5) / SP_K_SEG
    r_ = np.where(gated, 0.0, ch_r[w['segid']]); s_ = np.where(gated, 0.0, ch_s[w['segid']])
    cols  = [np.cumsum(np.where(bstep == b, r_, 0.0)) for b in range(SP_NBIN)]
    cols += [np.cumsum(np.where(bstep == b, s_, 0.0)) for b in range(SP_NBIN)]
    cols += [np.cumsum(0.5 * (r_ + s_) * np.sqrt(pos))]
    return np.column_stack(cols), Dr, dd

def _sp_kappa(donor_wids, Fr, Fs):
    """kappa on train(f) with per-donor SELF-EXCLUSION (audit-cell construction)."""
    A = y = None
    for w in (SP_WELL[d] for d in donor_wids):
        G = _sp_design_T2(w, Fr, Fs, (w['wi'],))[0]
        A = G.T @ G if A is None else A + G.T @ G
        y = G.T @ w['drift'] if y is None else y + G.T @ w['drift']
    return np.linalg.lstsq(A, y, rcond=None)[0]

def _sp_theta_from(wids):
    """Regional dip azimuth theta_f from the given wells ONLY (uses TVT+Z = labels ->
    must be train(f)-only; the L1 leak fix). Uses ANCC samples cached at load."""
    S = np.vstack([SP_WELL[w]['ancc_samp'] for w in wids])
    S = S[np.isfinite(S[:, 2])]
    Xg = np.column_stack([np.ones(len(S)), S[:, 0] - S[:, 0].mean(), S[:, 1] - S[:, 1].mean()])
    gb = np.linalg.lstsq(Xg, S[:, 2], rcond=None)[0]
    return float(np.arctan2(gb[2], gb[1]))

# ---- donors: every train well (SP_WELL carries the training-loader fields the engine reads) ----
print('building donor field from train/ ...')
_tr_files = sorted(glob.glob(os.path.join(TRAIN_DIR_DONORS, '*__horizontal_well.csv')))
SP_WELL = {}
for _p in _tr_files:
    wid = os.path.basename(_p).split('__')[0]
    hw = pd.read_csv(_p, usecols=['MD', 'X', 'Y', 'Z', 'TVT', 'TVT_input'])
    X, Y, Z = hw['X'].to_numpy(float), hw['Y'].to_numpy(float), hw['Z'].to_numpy(float)
    tvt, ti = hw['TVT'].to_numpy(float), hw['TVT_input'].to_numpy(float)
    fin = np.where(np.isfinite(ti))[0]
    if len(fin) == 0 or fin.max() >= len(tvt) - 8: continue
    sidx = int(fin.max()); ndz = -np.diff(Z)[sidx:]; n = len(ndz)
    drift = (tvt[sidx + 1:] - tvt[sidx]) - np.cumsum(ndz)
    st = max(len(hw) // 120, 1)
    SP_WELL[wid] = dict(wid=wid, s=sidx, n=n, X=X, Y=Y, drift=drift,
                        ancc_samp=np.column_stack([X[::st], Y[::st], (tvt + Z)[::st]]))
SP_THETA = _sp_theta_from(sorted(SP_WELL))
assert 95.0 < np.degrees(SP_THETA) < 125.0, f'theta {np.degrees(SP_THETA):.1f} deg out of band'
for i, wid in enumerate(sorted(SP_WELL)):
    w = SP_WELL[wid]; w['wi'] = i
    w['segid'], w['mid'], w['az'] = _sp_seg_geom(w['X'], w['Y'], w['s'], w['n'])
    w['proj'] = np.cos(w['az'] - SP_THETA)
    w['c_raw'] = _sp_fit_c(w['drift'], w['n'], 0.0)
    w['c_sm']  = _sp_fit_c(w['drift'], w['n'], SP_RHO)
SP_FR, SP_FS = _sp_build_field(sorted(SP_WELL), 'c_raw'), _sp_build_field(sorted(SP_WELL), 'c_sm')
SP_KAPPA = _sp_kappa(sorted(SP_WELL), SP_FR, SP_FS)
print(f'donor field: {len(SP_WELL)} wells, {len(SP_FR)} samples | theta {np.degrees(SP_THETA):.1f} deg '
      f'| |kappa| {np.linalg.norm(SP_KAPPA):.4f}   [training fingerprints: ~109.2 deg / 2.9474 at 773 donors]')

def spatial_rows_for_test_well(hw):
    """From a TEST hw frame (labels absent): geometry-only prior, returned as ROW-level arrays
    the image builder bins per (possibly phase-cropped) render:
      sp_prior_tvt (n_rows,): the prior's TVT per post-PS row (anchor + cumsum(ndz) + G @ kappa);
                              NaN on pre-PS rows (no bump there — the training convention).
      sp_logdd    (n_rows,): log1p(donor distance) per row via the segment map (sigma/amp inputs).
    Wells without usable post-PS geometry -> (None, None) == zero bump channel (neutral)."""
    X, Y, Z = hw['X'].to_numpy(float), hw['Y'].to_numpy(float), hw['Z'].to_numpy(float)
    ti = hw['TVT_input'].to_numpy(float)
    fin = np.where(np.isfinite(ti))[0]
    if len(fin) == 0 or fin.max() >= len(ti) - 8: return None, None
    sidx = int(fin.max()); ndz = -np.diff(Z)[sidx:]; n = len(ndz)
    w = dict(s=sidx, n=n)
    w['segid'], w['mid'], w['az'] = _sp_seg_geom(X, Y, sidx, n)
    w['proj'] = np.cos(w['az'] - SP_THETA)
    try:
        G, _, dd = _sp_design_T2(w, SP_FR, SP_FS)          # test wells are not in the field
        prior = G @ SP_KAPPA
    except Exception as _e:
        print(f'  spatial prior failed ({_e}); zero bump for this well'); return None, None
    n_rows = len(ti)
    sp_prior_tvt = np.full(n_rows, np.nan, dtype=np.float64)
    sp_prior_tvt[sidx + 1:] = float(ti[sidx]) + np.cumsum(ndz) + prior
    sp_logdd = np.full(n_rows, np.nan, dtype=np.float64)
    sp_logdd[sidx + 1:] = np.log1p(dd)[w['segid']]
    return sp_prior_tvt, sp_logdd


In [ ]:
# Resolve checkpoint paths + a per-batch loader for BATCHED ensembling.
# OOM FIX: the 35 stacked EfficientNetV2-L checkpoints do not fit in memory at once (weights alone
# exceed the 2x T4 32 GB, and even CPU-resident all-35 strains host RAM). Instead we resolve all paths
# up front (cheap) and load MODELS_PER_BATCH checkpoints from disk at a time inside the prediction
# loop, accumulate per-well prediction sums, and FREE each batch before loading the next. The ensemble
# averages at the expected-row level and every downstream step (decode, phase-broadcast, fill) is
# LINEAR, so batched accumulation is identical to loading all 35 at once (up to float summation order).
# build_model / the load audit / GPU sharding are unchanged.
import gc
_BN_COUNTER = 'num_batches_tracked'   # int BN counters; eval-irrelevant, absent from float-only EMA shadow

# How many checkpoints to hold in memory simultaneously. 35 ckpts on 2x T4 -> 6 spans both GPUs
# (~3/GPU). REDUCE if you still OOM; raise if there is headroom (fewer disk-load passes = faster).
MODELS_PER_BATCH = 8

# Resolve (and validate existence of) every checkpoint path WITHOUT loading weights.
ckpt_paths = []
for entry in manifest['checkpoints']:
    ckpt_path = os.path.join(CKPT_DIR, entry['filename'])
    if not os.path.exists(ckpt_path):
        # REUSE_PT saves ckpt_f{f}_s{s}_*.pt; the manifest omits the suffix.
        _hits = sorted(glob.glob(os.path.join(CKPT_DIR, entry['filename'][:-3] + '_*.pt')))
        if not _hits:
            raise FileNotFoundError(
                f"No checkpoint matching {entry['filename'][:-3]}_*.pt in {CKPT_DIR}. "
                f"Dir contains: {sorted(os.listdir(CKPT_DIR))[:10]}")
        ckpt_path = _hits[0]
    ckpt_paths.append(ckpt_path)
assert len(ckpt_paths) == len(manifest['checkpoints']), 'resolved path count != manifest count'

def _load_model_batch(paths):
    """Load a list of checkpoints into eval models, sharded across GPUs. Fails LOUD on arch mismatch
    (an unexpected/missing key means the inference architecture != the artifact)."""
    batch = []
    for j, ckpt_path in enumerate(paths):
        dev = GPUS[j % len(GPUS)] if SHARD_MODELS else CKPT_DEVICE
        state = torch.load(ckpt_path, map_location='cpu')   # host first, then place (avoids transient 2x on target)
        m = build_model(ENCODER_NAME, IN_CH).to(dev)
        missing, unexpected = m.load_state_dict(state, strict=False)
        if len(unexpected) != 0:
            raise RuntimeError(
                f"{os.path.basename(ckpt_path)}: {len(unexpected)} checkpoint keys did not map into the "
                f"model (e.g. {list(unexpected)[:4]}). Inference architecture does not match the artifact "
                f"-- check STACK_N ({STACK_N}), encoder ({ENCODER_NAME}), in_ch ({IN_CH}).")
        bad_missing = [k for k in missing if not k.endswith(_BN_COUNTER)]
        if len(bad_missing) != 0:
            raise RuntimeError(
                f"{os.path.basename(ckpt_path)}: {len(bad_missing)} non-BN params missing from the "
                f"checkpoint (e.g. {bad_missing[:4]}). Artifact is incomplete for this architecture.")
        m.eval()
        batch.append(m)
        del state
    return batch

def _free_model_batch(batch):
    """Drop a batch's models and reclaim GPU + host memory before the next batch loads."""
    for m in batch:
        del m
    batch.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

_arch = 'StackedUNet' if STACK_N >= 2 else 'UNetGeneric'
_n_batches = (len(ckpt_paths) + MODELS_PER_BATCH - 1) // MODELS_PER_BATCH
print(f'Resolved {len(ckpt_paths)} checkpoints for {_arch} (encoder={ENCODER_NAME}, in_ch={IN_CH}'
      + (f', stage2_in_ch={IN_CH + 1}' if STACK_N >= 2 else '') + ').')
print(f'  Batched inference: {MODELS_PER_BATCH} ckpts/batch -> {_n_batches} batches '
      f'(loaded from disk per batch; never all {len(ckpt_paths)} resident at once).')
print(f'  Sharding: ' + ('round-robin across ' + str(len(GPUS)) + ' GPU(s)' if SHARD_MODELS else str(CKPT_DEVICE))
      + f'; compute device: {DEVICE}.')


In [ ]:
# === PhaseJitter TTA — WIRED AS IS (crop toe rows -> re-bin -> predict -> broadcast -> average
# in TVT space per MD row). The k-iterates share the stage-1 forward (k1/k2 free on the way to k3).
# CHANGES for the 5.0270 line, decode-side only: (a) the per-model decode is _gated_expected_row
# (softmax expectation + the LEARNED gate per depth — the training evaluate_model deployment path);
# (b) the DO_SPATIAL_BLEND per-seed streams are gone (blend removed). The phase machinery — offsets,
# crop, coverage accounting, toe fill — is byte-untouched. The spatial channel re-renders per phase
# crop automatically (row-level sp arrays truncate with every other (n_hw,) array).
DO_PHASE_JITTER_TTA = True
PHASE_TTA_OFFSETS = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875]  # full-bin sweep

def predict_well_streams(rec_arrays, models, model_seeds, do_tta, offsets):
    """Multi-stream phase-averaged per-MD-row predictions for ONE well and ONE model batch.
    Returns ({stream: pred_full (n_hw,)}, {stream: n_member_models_in_this_batch}).
    Streams: 'k{j}' (all batch models) for j=1..K_COMPUTE; 'k{EVAL_K}_s{seed}' (that seed's batch
    models) when the blend is on. Linearity: outer accumulation weights by member count, so batched
    == all-at-once (up to float summation order), exactly as before."""
    n_hw = len(rec_arrays['hw_md'])
    bin_w = n_hw / float(W)
    use_offsets = offsets if (do_tta and len(offsets) > 0) else [0.0]
    stream_names = [f'k{j}' for j in range(1, K_COMPUTE + 1)]
    members = {st: len(models) for st in stream_names}
    acc = {st: np.zeros(n_hw, dtype=np.float64) for st in stream_names}
    cnt = np.zeros(n_hw, dtype=np.float64)
    _stream_ckpts = str(CKPT_DEVICE) != str(DEVICE)
    for off in use_offsets:
        drop = int(round(float(off) * bin_w))
        drop = min(drop, n_hw - W - 1) if n_hw - W - 1 > 0 else 0
        end = n_hw - drop
        a = {k: (v[:end] if isinstance(v, np.ndarray) and v.shape[0] == n_hw else v)
             for k, v in rec_arrays.items()}
        try:
            rec = build_image_from_arrays(**a)
        except Exception:
            continue
        img_cpu = torch.from_numpy(rec['image_in']).unsqueeze(0)
        ers = {j: [] for j in range(1, K_COMPUTE + 1)}      # per-model expected rows, tagged by seed
        with torch.no_grad():
            for m, sd in zip(models, model_seeds):
                dev  = next(m.parameters()).device
                mm   = m.to(DEVICE) if _stream_ckpts else m
                cdev = DEVICE if _stream_ckpts else dev
                x    = img_cpu.to(cdev, non_blocking=True)
                kl   = _iter_logits(mm, x, K_COMPUTE)
                for j in range(1, K_COMPUTE + 1):
                    # deployment decode: expectation + the LEARNED gate at THIS depth (== training eval)
                    er = _gated_expected_row(mm, x, kl[j])[0].cpu().numpy()
                    ers[j].append((sd, er))
                if _stream_ckpts: m.to(CKPT_DEVICE)
        be = rec['bin_edges']
        for st in stream_names:
            sel = [er for (_, er) in ers[int(st[1:])]]
            if not sel: continue
            pred_col = rec['tvt_lo'] + (np.stack(sel, axis=0).mean(axis=0) + 0.5) * BIN_SIZE
            for c in range(W):
                acc[st][be[c]:be[c + 1]] += pred_col[c]
        for c in range(W):                                   # coverage identical across streams
            cnt[be[c]:be[c + 1]] += 1.0
    preds = {}
    covered = cnt > 0
    for st in stream_names:
        if members[st] == 0: continue
        p = np.full(n_hw, np.nan, dtype=np.float64)
        p[covered] = acc[st][covered] / cnt[covered]
        if not covered.all():                                # fill uncovered toe rows forward
            last = np.nan
            for r in range(n_hw):
                if covered[r]: last = p[r]
                elif not np.isnan(last): p[r] = last
            if np.isnan(p[0]):
                p[:np.argmax(covered)] = p[covered][0] if covered.any() else 0.0
        preds[st] = p
    return preds, members

print(f'PhaseJitter TTA: {"ON" if DO_PHASE_JITTER_TTA else "OFF (single-pass)"}'
      + (f' — offsets {PHASE_TTA_OFFSETS}' if DO_PHASE_JITTER_TTA else '')
      + f' | streams: k1..k{K_COMPUTE} | decode: gated expectation (in-model)')


In [ ]:
# Walk test wells: multi-stream BATCHED ensembling. The spatial channel arrives via the per-well
# ROW-level arrays cached in pass 1 (they ride the rec_arrays dict into every phase-cropped render).
test_hw_files = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
print(f'Found {len(test_hw_files)} test wells')

# === dZ sanity (unchanged from the pre-fork kernel) ===
_check_hw = pd.read_csv(test_hw_files[0])
assert 'Z' in _check_hw.columns, f"Test HW CSV has no 'Z' column. Columns: {list(_check_hw.columns)}"
_post = _check_hw['TVT_input'].isna().values
_zpost = _check_hw['Z'].values[_post]
_n_finite = int(np.isfinite(_zpost).sum())
_z_std = float(np.nanstd(_zpost)) if _n_finite > 1 else 0.0
print(f"[Z sanity] first test well: {len(_zpost)} post-PS rows, {_n_finite} finite Z, std={_z_std:.3f}")
if _n_finite == 0:
    raise RuntimeError('Post-PS Z entirely missing in test — dZ channel degenerate; do not submit this model.')
if _z_std < 1e-6:
    print('  WARNING: post-PS Z ~constant — dZ channel ~flat.')

# checkpoint seeds, parallel to ckpt_paths (manifest is the source of truth)
ckpt_seeds = [int(entry['seed']) for entry in manifest['checkpoints']]
assert len(ckpt_seeds) == len(ckpt_paths)

# --- Pass 1: cache raw arrays + geometry + (blend) the spatial prior per well -----------------
well_recs = {}; well_order = []; post_idx = {}; last_known_tvt = {}
_n_sp = 0
for hw_path in test_hw_files:
    well_id = os.path.basename(hw_path).split('__')[0]
    tw_path = hw_path.replace('__horizontal_well.csv', '__typewell.csv')
    if not os.path.exists(tw_path):
        print(f'  {well_id}: MISSING typewell CSV, skipping'); continue
    hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path)
    _sp_tvt, _sp_ld = spatial_rows_for_test_well(hw)       # the CHANNEL's row-level inputs
    if _sp_tvt is not None: _n_sp += 1
    well_recs[well_id] = dict(
        hw_md=hw['MD'].values, hw_gr=hw['GR'].values, hw_z=hw['Z'].values,
        hw_tvt=np.full(len(hw), np.nan, dtype=np.float64), hw_tvti=hw['TVT_input'].values,
        tw_tvt=tw['TVT'].values, tw_gr=tw['GR'].values,
        sp_prior_tvt=_sp_tvt, sp_logdd=_sp_ld)
    well_order.append(well_id)
    post_idx[well_id] = np.where(hw['TVT_input'].isna().values)[0]
    known = hw['TVT_input'].dropna()
    last_known_tvt[well_id] = float(known.iloc[-1]) if len(known) > 0 else float('nan')
print(f'spatial channel built for {_n_sp}/{len(well_order)} wells (others: zero bump, model-only)')
print(f'Cached {len(well_order)} wells; ensembling {len(ckpt_paths)} ckpts in batches of {MODELS_PER_BATCH}.')

# --- Pass 2: per-stream accumulation across model batches -------------------------------------
stream_sum = {w: {} for w in well_order}     # well -> stream -> weighted pred sum
stream_n   = {}                              # stream -> total member models accumulated
for b in range(_n_batches):
    lo = b * MODELS_PER_BATCH
    paths  = ckpt_paths[lo:lo + MODELS_PER_BATCH]
    seeds  = ckpt_seeds[lo:lo + MODELS_PER_BATCH]
    batch_models = _load_model_batch(paths)
    members_seen = None
    for well_id in well_order:
        preds, members = predict_well_streams(well_recs[well_id], batch_models, seeds,
                                              DO_PHASE_JITTER_TTA, PHASE_TTA_OFFSETS)
        members_seen = members
        for st, p in preds.items():
            prev = stream_sum[well_id].get(st)
            stream_sum[well_id][st] = p * members[st] if prev is None else prev + p * members[st]
    for st, mcount in members_seen.items():
        stream_n[st] = stream_n.get(st, 0) + mcount
    _free_model_batch(batch_models)
    print(f'  batch {b + 1}/{_n_batches}: +{len(batch_models)} ckpts '
          f'({sum(stream_n[st] for st in stream_n if st == f"k{EVAL_K}")}/{len(ckpt_paths)} on k{EVAL_K}), freed')
assert stream_n[f'k{EVAL_K}'] == len(ckpt_paths)

# --- Finalize: per-stream means; the k{EVAL_K} stream is THE model prediction ------------------
STREAMS = {w: {st: stream_sum[w][st] / float(stream_n[st]) for st in stream_sum[w]} for w in well_order}

predictions = {}
for well_id in well_order:
    pred_final = STREAMS[well_id][f'k{EVAL_K}']              # the gated k3 stream IS the deliverable
    for ri in post_idx[well_id]:
        predictions[(well_id, int(ri))] = float(pred_final[ri])

print(f'\nTotal predictions: {len(predictions):,}  (k={EVAL_K} refeed, {len(ckpt_paths)} ckpts'
      + (', TTA' if DO_PHASE_JITTER_TTA else '') + ', in-model spatial channel + gate)')


In [ ]:
# Build submission in the order specified by sample_submission.csv
sample_sub = pd.read_csv(SAMPLE_SUB)
print(f'Sample submission has {len(sample_sub):,} rows, columns: {sample_sub.columns.tolist()}')

# Drop the placeholder tvt column from the sample so it doesn't shadow our predictions in the merge.
# (Earlier bug: with suffixes=('', '_pred'), the left tvt column kept its name and our predictions
#  ended up in tvt_pred. Sample-submission zeros were then written to submission.csv.)
sample_sub_ordered = sample_sub[['id']].copy()

# Parse well_id and md_row from id column (id format: '{well_id}_{md_row_0_based}')
parts = sample_sub_ordered['id'].str.rsplit('_', n=1, expand=True)
sample_sub_ordered['well_id'] = parts[0]
sample_sub_ordered['md_row']  = parts[1].astype(int)

# Convert the predictions dict to a DataFrame for a fast merge
pred_records = [{'well_id': w, 'md_row': r, 'tvt': v} for (w, r), v in predictions.items()]
pred_df = pd.DataFrame(pred_records)
print(f'Predictions DataFrame: {len(pred_df):,} rows')

# Merge predictions into submission order (no column conflict now, tvt comes from pred_df)
merged = sample_sub_ordered.merge(pred_df, on=['well_id', 'md_row'], how='left')

# Apply fallback for any unmatched rows (shouldn't happen if test wells match the submission template)
n_missing = int(merged['tvt'].isna().sum())
if n_missing > 0:
    print(f'  {n_missing} rows had no model prediction; filling with per-well last-known TVT')
    fallback_map = pd.Series(last_known_tvt)
    merged.loc[merged['tvt'].isna(), 'tvt'] = merged.loc[merged['tvt'].isna(), 'well_id'].map(fallback_map)

n_still_nan = int(merged['tvt'].isna().sum())
if n_still_nan > 0:
    print(f'  {n_still_nan} rows still NaN after fallback; filling with global mean')
    merged['tvt'] = merged['tvt'].fillna(merged['tvt'].mean())

# Sanity checks
n_total = len(merged)
n_from_model = n_total - n_missing
print(f'\nSubmission rows: {n_total:,}')
print(f'  From model predictions: {n_from_model:,} ({100*n_from_model/n_total:.2f}%)')
print(f'  From fallback: {n_missing:,} ({100*n_missing/n_total:.2f}%)')

# Hard sanity check: predictions shouldn't be all zero (catches the bug we just fixed)
if merged['tvt'].abs().max() < 1.0:
    raise RuntimeError(
        f'Sanity check FAILED: max |tvt| < 1.0 across all rows. '
        f'Predictions are essentially zero. Inspect the merge.'
    )

# Write submission with the original columns only
out = merged[['id', 'tvt']]
out.to_csv('submission.csv', index=False)
print(f'\nWrote submission.csv with {len(out):,} rows')
print(f'\nFirst rows:')
print(out.head().to_string())
print(f'\nLast rows:')
print(out.tail().to_string())
print(f'\nPrediction stats: min={out["tvt"].min():.2f}, mean={out["tvt"].mean():.2f}, max={out["tvt"].max():.2f}')



In [ ]:
# Quick visual sanity check: per-well prediction trajectories
import matplotlib.pyplot as plt

n_wells_to_plot = min(3, len(test_hw_files))
fig, axes = plt.subplots(1, n_wells_to_plot, figsize=(5 * n_wells_to_plot, 4))
if n_wells_to_plot == 1:
    axes = [axes]

for ax, hw_path in zip(axes, test_hw_files[:n_wells_to_plot]):
    well_id = os.path.basename(hw_path).split('__')[0]
    hw = pd.read_csv(hw_path, usecols=['MD', 'TVT_input'])
    is_post_ps = hw['TVT_input'].isna().values
    md_indices_post = np.where(is_post_ps)[0]
    md_indices_pre  = np.where(~is_post_ps)[0]
    preds = np.array([predictions.get((well_id, int(ri)), np.nan) for ri in md_indices_post])

    ax.plot(hw.loc[md_indices_pre, 'MD'].values, hw.loc[md_indices_pre, 'TVT_input'].values,
            'b-', lw=1.5, label='TVT_input (known)')
    ax.plot(hw.loc[md_indices_post, 'MD'].values, preds,
            'r-', lw=1.5, label='FT39e ens prediction')

    last_known = last_known_tvt.get(well_id, np.nan)
    if not np.isnan(last_known):
        ax.axhline(last_known, color='gray', ls=':', alpha=0.5, label='last known value')
    ax.set_title(f'{well_id}: {len(md_indices_post)} predicted rows')
    ax.set_xlabel('MD'); ax.set_ylabel('TVT')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Sanity: predicted values should be in a sensible range relative to last-known
print('\nPer-well prediction sanity:')
for hw_path in test_hw_files[:5]:
    well_id = os.path.basename(hw_path).split('__')[0]
    last_known = last_known_tvt.get(well_id, np.nan)
    preds_well = [v for (w, _), v in predictions.items() if w == well_id]
    if preds_well:
        arr = np.array(preds_well)
        print(f'  {well_id}: last known TVT = {last_known:.2f}, '
              f'pred range = [{arr.min():.2f}, {arr.max():.2f}], '
              f'pred mean = {arr.mean():.2f}, deviation from last = {arr.mean() - last_known:+.2f}')
